<a href="https://colab.research.google.com/github/Mr-KapilAgnihotri/Playing_with_datasets/blob/main/Text_Moderation_filter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install better_profanity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 1.9 MB/s eta 0:00:00


In [6]:
from better_profanity import profanity

# 1. Initialize the default dictionary of bad words
profanity.load_censor_words()

# You can also add your own custom words to the blacklist easily!
profanity.add_censor_words(['heck', 'darn', 'nigga','dicks'])

# 2. Get input directly in your Jupyter Notebook
user_text = input("Type a comment to test: ")

# 3. Process the text
# This checks the text and replaces bad words with ****
masked_text = profanity.censor(user_text)
is_clean = not profanity.contains_profanity(user_text)

# 4. Display the results like your pipeline would
print("\n" + "="*30)
print(" MODERATION PIPELINE RESULT")
print("="*30)
print(f"Original Text (DB): {user_text}")
print(f"Displayed Text    : {masked_text}")
print(f"Is Moderated Flag : {not is_clean}")

Type a comment to test: fuck you asshole

 MODERATION PIPELINE RESULT
Original Text (DB): fuck you asshole
Displayed Text    : **** you ****
Is Moderated Flag : True


In [7]:
!pip install detoxify

In [9]:
from detoxify import Detoxify

#1. Load the model
print("Loading Hugging Face BERT Model...")
model = Detoxify('original')



Loading Hugging Face BERT Model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
# 2. Get input
user_text = input("Type a comment to test: ")

# 3. Run the ML prediction
print("\nAnalyzing context...")
results = model.predict(user_text)

# 4. Display the ML Confidence Scores
print("\n" + "="*30)
print(" ML TOXICITY SCORES (0 to 1)")
print("="*30)
for category, score in results.items():
    print(f"{category.capitalize():<18}: {score:.4f}")

# 5. Your Pipeline Logic
# Let's say if any score is over 70% (0.70), we flag the post
THRESHOLD = 0.70
is_toxic = any(score > THRESHOLD for score in results.values())

print("\n--- PIPELINE DECISION ---")
if is_toxic:
    print("🚨 STATUS: REJECTED (Update DB: is_moderated = true)")
    # In your real app, you would decide if you want to delete the post,
    # hide it, or pass it to Option 1 to mask the specific words.
else:
    print("✅ STATUS: APPROVED (Update DB: is_moderated = false)")

Type a comment to test: how you feeling mother fuckin nigga you asshole looking like pussy, but you surea are a dick faced fucker

Analyzing context...

 ML TOXICITY SCORES (0 to 1)
Toxicity          : 0.9988
Severe_toxicity   : 0.6798
Obscene           : 0.9917
Threat            : 0.0366
Insult            : 0.9830
Identity_attack   : 0.8095

--- PIPELINE DECISION ---
🚨 STATUS: REJECTED (Update DB: is_moderated = true)


In [11]:
!pip install detoxify better_profanity

In [12]:
from detoxify import Detoxify
from better_profanity import profanity

In [13]:
print("Loading Models...")
ml_model = Detoxify('original')
profanity.load_censor_words()

Loading Models...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:

# 2. Get user input
user_text = input("Type a comment to test: ")

# 3. Step One: The ML Gatekeeper
print("\nRunning ML Context Check...")
results = ml_model.predict(user_text)

# We define toxicity as any category scoring over 70%
THRESHOLD = 0.70
is_toxic = any(score > THRESHOLD for score in results.values())

# 4. Pipeline Logic for your Database
original_text = user_text
display_text = user_text
is_moderated = False

if is_toxic:
    is_moderated = True

    # Step Two: Try to mask specific words first
    masked_attempt = profanity.censor(user_text)

    # Check if the dictionary actually found words to mask
    if masked_attempt != user_text:
        # It found specific bad words! (e.g., "You are a ****")
        display_text = masked_attempt
    else:
        # It was toxic, but no specific bad words were used (e.g., a threat).
        # We must mask the entire text to protect the community.
        display_text = "[This comment was removed by moderators for violating community guidelines]"

# 5. Display the final result for your Database Update
print("\n" + "="*40)
print(" FINAL DATABASE PAYLOAD (Send to Kafka)")
print("="*40)
print(f"originalText : {original_text}")
print(f"displayText  : {display_text}")
print(f"isModerated  : {is_moderated}")

Type a comment to test: do you want to die you asshole?

Running ML Context Check...

 FINAL DATABASE PAYLOAD (Send to Kafka)
originalText : do you want to die you asshole?
displayText  : do you want to die you ****?
isModerated  : True
